# Setup

## Installation

In [31]:
! pip install 'smolagents[toolkit]'
! pip install litellm

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

## HuggingFace Login

In [33]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

# Making Custom Tool

In [34]:
from smolagents import tool
from huggingface_hub import list_models

@tool
def model_download_tool(task: str) -> str:
  '''
  This is a tool that returns the most downloaded model of a given task on the hugging face hub.
  it returns name of the checkpoint

  Args:
      task: the task for which to get the download count
  '''
  most_downloaded_model = next(iter(list_models(
      filter = task,
      sort = "downloads",
      direction = -1
      )
  )
  )

  return most_downloaded_model.id


## Let's give this tool to an Agent

In [21]:
from smolagents import CodeAgent, InferenceClientModel
from smolagents.models import LiteLLMModel

model = LiteLLMModel(
    model_id="groq/llama-3.1-8b-instant",
    api_key=userdata.get('GROQ_TOKEN')
)
# Create an agent with no tools
agent = CodeAgent(
    tools=[model_download_tool],
    model=model
    )

# Run the agent with a task
prompt = "Give me the most downloaded model for text-generation on the Hugging Face Hub."
result = agent.run(prompt)
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me the most downloaded model for text-generation on the Hugging Face Hub.                                  │
│                                                                                                                 │
╰─ LiteLLMModel - groq/llama-3.1-8b-instant ──────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  task = "text-generation"                                                                                         
  most_downloaded_model_name = model_download_tool(task)                                                           
  final_answer(most_downloaded_model_name)                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: openai-community/gpt2

[Step 1: Duration 1.10 seconds| Input tokens: 2,082 | Output tokens: 1]

openai-community/gpt2


# Adding a tool to convert text to image

In [28]:
from smolagents import Tool
from huggingface_hub import InferenceClient

class TextToImageTool(Tool):
    description = "This tool creates an image according to a prompt, which is a text description."
    name = "image_generator"
    inputs = {
        "prompt": {
            "type": "string",
            "description": "The image generator prompt. Don't hesitate to add details in the prompt to make the image look better, like 'high-res, photorealistic', etc."
        },
        "model": {
            "type": "string",
            "description": "The Hugging Face model ID to use for image generation. If not provided, will use the default model."
        }
    }
    output_type = "image"
    current_model = "black-forest-labs/FLUX.1-schnell"

    def forward(self, prompt, model):
        # self.client = InferenceClient(self.current_model)
        if model:
            if model != self.current_model:
                self.current_model = model
                self.client = InferenceClient(model)
        if not self.client:
            self.client = InferenceClient(self.current_model)

        image = self.client.text_to_image(prompt)
        image.save("image.png")

        return f"Successfully saved image with this prompt: {prompt} using model: {self.current_model}"

In [39]:
from smolagents import InferenceClientModel, DuckDuckGoSearchTool

image_generator = TextToImageTool()
model_id = "Qwen/QwQ-32B-Preview"
GROQ_MODEL_ID = "groq/llama-3.3-70b-versatile" #"groq/llama-3.1-8b-instant"

model = LiteLLMModel(
    model_id=GROQ_MODEL_ID,
    api_key=userdata.get('GROQ_TOKEN'),
    max_tokens=512,
    num_retries=5
)

agent = CodeAgent(
    tools=[image_generator, model_download_tool, DuckDuckGoSearchTool()],
    model=model
    )
agent.run(
    "Improve this prompt, then generate an image of it. Prompt: A cat wearing a hazmat suit in contaminated area.  Get the latest model for text-to-image from the Hugging Face Hub."
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Improve this prompt, then generate an image of it. Prompt: A cat wearing a hazmat suit in contaminated area.    │
│ Get the latest model for text-to-image from the Hugging Face Hub.                                               │
│                                                                                                                 │
╰─ LiteLLMModel - groq/llama-3.3-70b-versatile ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  latest_model = model_download_tool(task="text-to-image")                                                         
  print("Latest model for text-to-image:", latest_model)                                                           
  improved_prompt = "A high-res, photorealistic cat wearing a hazmat suit in a highly contaminated area with       
  warning signs and toxic waste"                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Latest model for text-to-image: stable-diffusion-v1-5/stable-diffusion-v1-5

Out: A high-res, photorealistic cat wearing a hazmat suit in a highly contaminated area with warning signs and 
toxic waste

[Step 1: Duration 1.43 seconds| Input tokens: 2,255 | Output tokens: 100]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider L

Error in generating model output:
litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model 
`llama-3.3-70b-versatile` in organization `org_01k53r2xt1ewws41wx1fhxwrwh` service tier `on_demand` on tokens per 
minute (TPM): Limit 12000, Used 10168, Requested 3645. Please try again in 9.064s. Need more tokens? Upgrade to Dev
Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

[Step 5: Duration 0.47 seconds]

AgentGenerationError: Error in generating model output:
litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01k53r2xt1ewws41wx1fhxwrwh` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 10168, Requested 3645. Please try again in 9.064s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
